[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RickBarretto/llm-playground/blob/main/rodar.ipynb)


## Configuração

Este notebook instala `arcadia` para executar modelos de IA.

Antes de executar um modelo, habilite um runtime com GPU no Colab: `Runtime` -> `Change runtime type` -> `T4 GPU` ou melhor.

Defina `GIT_REF` como a branch ou tag que deseja testar. Exemplos: `main`, `v1.0.0`.

In [ ]:
GIT_REF = "main"  # branch ou tag

!rm -rf /content/arcadia
!git clone https://github.com/LasicUEFS/arcadia /content/arcadia
%cd /content/arcadia
!git fetch --all --tags
!git checkout $GIT_REF
!pip install -U /content/arcadia

## Diretório De Saída

Respostas, poemas, avaliações e documentos gerados devem ser armazenados fora deste repositório.

Prefira um repositório separado dentro de um cofre do Obsidian ao trabalhar localmente. No Colab, mantenha as saídas fora de `/content/arcadia` e baixe ou sincronize os arquivos a partir dali.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Definir Modelo

|Modelo|Tamanhos Disponíveis|Tamanho Padrão|Hugging Face|
|:---:|:-------------------|:------------:|------------|
|`AmadeusVerbo`  |`0.5B`, `1.5B`, `3B`, `7B`, `14B`, `32B`, `72B`|`7B`|https://huggingface.co/collections/amadeusai/amadeus-verbo-qwen25-pt-br-powered-by-aws|
|`Gaia`          |`4B`|`4B`|https://huggingface.co/CEIA-UFG/Gemma-3-Gaia-PT-BR-4b-it|
|`Tucano`        |`1.1B`, `2.4B`| `2.4B`|https://huggingface.co/collections/TucanoBR/tucano|
|`TeenyTinyLlama`|`460m`| `460m`|https://huggingface.co/collections/nicholasKluge/teenytinyllama|

**Uso:**

```python
from arcadia import models

model = models.AmadeusVerbo("3B")
```

In [ ]:
from arcadia import models

model = models.AmadeusVerbo("3B")

## Perguntar

`ask()` retorna a resposta do modelo como texto UTF-8 normalizado quando impressa ou gravada em disco.

In [ ]:
with model as m:
    output = m.ask("Escreva uma frase curta sobre o Brasil.")
    print(output)
    (OUTPUT_DIR / "ask.txt").write_text(output, encoding="utf-8")

## Chat

`chat()` produz rodadas em um formato natural:

```text
Usuário:

...

<Modelo>:

...
```

Use `sair`, `exit` ou `quit` para finalizar um chat interativo.

In [ ]:
with model as m:
    output = None

    for output in m.chat():
        print(output)
        print()

    if output is not None:
        (OUTPUT_DIR / "chat-last.txt").write_text(output, encoding="utf-8")

## Histórico Do Chat

Um histórico de chat pode ser indexado. `history[-1]` imprime apenas a última rodada usuário/modelo.

Use `history[-1].ai` ou `history.last.ai` para obter apenas a resposta do modelo.
Se quiser ver sua pergunta, use `history[index].user` ou `history[index].me`.

In [ ]:
with model as m:
    history = m.chat([
        "Escreva uma frase sobre o Brasil.",
        "Agora transforme essa frase em um verso."
    ])

    print(history[-1])
    print(history[-1].ai)
    print(history.last.ai)
    (OUTPUT_DIR / "chat-history-last.txt").write_text(history[-1], encoding="utf-8")

## Limpeza

Se quiser reutilizar o mesmo runtime, libere o modelo antes de carregar outro.

In [ ]:
model.release()
del model